In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.utils.data import DataLoader, random_split
import numpy as np

In [2]:
torch.manual_seed(1)
SEED = 1
BATCH_SIZE = 8
IMG_SIZE = 32

In [ ]:
transform = transforms.ToTensor()

# download data
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
# split 8:2 for val set
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# create dataloader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
# assume image (32,32,3)
# maybe dropout here too
# Img to patch
class to_patch(nn.Module):
    def __init__(self, patch_size, embedding_dim):
        # patch_size 1, 2, 4, 8, 16 == patch width or height
        super().__init__()
        self.patch_size = patch_size
        self.channel_size = 3

        self.patch_count = (IMG_SIZE // patch_size)**2
        self.patch_dim = self.channel_size * self.patch_size**2

        self.projection = nn.Linear(self.patch_dim, embedding_dim)

    def forward(self, x):
        b, c, h, w = x.shape
        n = self.patch_count
        p = self.patch_size

        # To make (B, C, H, W) -> (B, N, P^2*C)
        # change order to make patch in same bucket
        x = x.reshape(b, c, h//p, p, w//p, p)
        x = x.permute(0, 2, 4, 1, 3, 5)

        x = x.reshape(b, n, -1)

        x = self.projection(x)

        return x

# positional encoding from hw
def sinusoidal_positions(length: int, model_dim: int) -> np.ndarray:
    if model_dim%2 != 0:
        raise ValueError("model_dim should be even")

    pe = np.zeros((length, model_dim))

    m = np.arange(model_dim/2)
    p = np.arange(length).reshape(-1, 1)

    wm = np.power(10000., -(2*m)/model_dim)

    pe[:, 0::2] = np.sin(p*wm)
    pe[:, 1::2] = np.cos(p*wm)

    return pe

# Add pos encoding and CLS token
class preprocess(nn.Module):
    def __init__(self, patch_size, embedding_dim):
        super().__init__()
        self.patch_count = (IMG_SIZE//patch_size)**2
        self.cls_token = nn.Parameter(torch.zeros(1, 1,embedding_dim))

        pos_embedding = sinusoidal_positions(self.patch_count+1, embedding_dim)
        pos_embedding = torch.tensor(pos_embedding, dtype=torch.float32).unsqueeze(0)
        # gpt, to make pos_embedding not trainable
        self.register_buffer("pos_embedding", pos_embedding)

    def forward(self, x):
        batch = x.shape[0]

        # copy cls_token as number of batch
        cls_token = self.cls_token.expand(batch, -1, -1)

        # concat
        x = torch.cat((cls_token, x), dim=1)

        # pos encoding
        x = x + self.pos_embedding
        return x

Build transformer and classification head

In [ ]:
# transformer encoders
# input as embedding_dim=(Batch, token count, embedding size)
#----------------------------------
# input
# normalize (layer norm)
# MHA
# add input (residual connection)
# input2
# normalize (layer norm)
# MLP
# add input2 (residual connection)
#----------------------------------
class transformer_block(nn.Module):
    def __init__(self, embedding_dim, num_heads, hidden_dim, dropout=0):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.mha = nn.MultiheadAttention(embedding_dim,
                                         num_heads,
                                         dropout,
                                         batch_first=True)
        #self.dropout = nn.Dropout(dropout)

        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x1 = self.layer_norm1(x) # layer norm
        # x2, _ = self.mha(x1, x1, x1) # multi head self attention
        x2, _ = self.mha(x1, x1, x1, need_weights=False)
        x2 = x2 + x # residual connection

        x3 = self.layer_norm2(x2) # layer norm
        x3 = self.mlp(x3) # fc layer
        x3 = x3 + x2
        return x3

# multiple transformer blocks
class transformer_encoder(nn.Module):
    def __init__(self, num_blocks, embedding_dim, num_heads, hidden_dim, dropout=0):
        super().__init__()
        self.blocks = nn.ModuleList([
            transformer_block(embedding_dim, num_heads, hidden_dim, dropout) for _ in range(num_blocks)
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

#--------------------------------
# use first one as input for MLP
# predict
#--------------------------------
class classification_head(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_class=10, dropout=0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_class),
        )
    def forward(self, x):
        return self.mlp(x)

class vision_transformer(nn.Module):
    def __init__(self, to_patch, preprocess, encoder, head):
        super().__init__()
        self.to_patch = to_patch
        self.preprocess = preprocess
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        x = self.to_patch(x)
        x = self.preprocess(x)
        x = self.encoder(x)
        class_token = x[:, 0, :] # take first token for classification
        x = self.head(class_token)
        return x

In [ ]:
num_heads = 4
dropout=0
patch_sizes = [1, 2, 4, 8, 16] # patch width or height
embedding_dim = 384
num_blocks = 6
encoder_hidden_dim = 768
head_hidden_dim = 3072

# preprocess input
to_patch = to_patch(patch_size=patch_sizes[0],
                    embedding_dim=embedding_dim)
preprocess = preprocess(patch_size=patch_sizes[0],
                        embedding_dim=embedding_dim)

# model inifialization and forward pass
encoder = transformer_encoder(num_blocks=num_blocks,
                              embedding_dim=embedding_dim,
                              num_heads=num_heads,
                              hidden_dim=encoder_hidden_dim,
                              dropout=0)
head = classification_head(embedding_dim=embedding_dim,
                           hidden_dim=head_hidden_dim,
                           num_class=10,
                           dropout=0)

vit = vision_transformer(to_patch, preprocess, encoder, head)